# 2D Harrison non-product benchmark

This notebook contains the public fixed-endpoint experiment for the
two-dimensional Harrison reflected Brownian motion benchmark. The interior
joint log-transform combines a shared coordinate Holomorphic Cauchy branch
with a centered full-input Holomorphic Cauchy branch. The latter receives both
complex coordinates simultaneously and therefore removes the product-form
restriction.

The network has exactly 15,236 trainable real scalar degrees of freedom. The
formal protocol uses seed 12345, 15,000 optimizer updates, and 64.05 million
sampled constraints. Only the prespecified step-15,000 model is saved. Exact
tails and published NN4MGF predictions are introduced only after training and
cannot influence the loss, endpoint, or hyperparameters.

`RUN_MODE="smoke"` is the safe default. Use `RUN_MODE="train"` for the paper
protocol or `RUN_MODE="evaluate"` with an explicit clean run directory to
evaluate an existing checkpoint. The 2D comparison uses Talbot Degree 5 only;
predictions are raw and are never clipped.


## 1. Imports and reproducibility


In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import csv
import hashlib
import json
import math
import platform
import random
import sys
import time
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import mpmath as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

r_dtype = torch.float64
c_dtype = torch.complex128
torch.set_default_dtype(r_dtype)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.use_deterministic_algorithms(True, warn_only=True)


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cuda_sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2), encoding="utf-8")


def write_csv(path, rows, fieldnames):
    with Path(path).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def safe_torch_load(path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


print({"python": sys.version.split()[0], "torch": torch.__version__, "device": str(device)})

## 2. Frozen paper configuration


In [ ]:
# Public defaults are safe: Run All performs only a two-step smoke test.
# Set RUN_MODE="train" for the fixed 15,000-step paper experiment.
# Set RUN_MODE="evaluate" and EVALUATE_RUN_DIR for an existing clean run.
RUN_MODE = "smoke"                  # "smoke", "train", or "evaluate"
EVALUATE_RUN_DIR = None             # e.g., Path("outputs/d2_harrison_...")
OUTPUT_ROOT = Path("outputs")

EXPERIMENT = "d2_harrison_nonproduct_global_cauchy"
SEED = 12345
D = 2
TRAIN_STEPS = 15_000
PRIMARY_TALBOT_DEGREE = 5
T_VALUES = list(range(1, 6))
TARGET_TRAINABLE_REAL_SCALAR_DOF = 15_236
POLE_TOL = 1e-12

MODEL_CONFIG = {
    "d": D,
    "coord_embed_dim": 16,
    "boundary_embed_dim": 16,
    "interior_hidden_width": 95,
    "boundary_hidden_width": 128,
    "global_hidden_width": 109,
    "cauchy_layers_per_scalar_net": 1,
    "pair_chunk_size": 32768,
    "boundary_anchor_mode": "hard_rbm_flow_balance",
    "full_input_global_branch": True,
    "nonproduct_global_branch": True,
    "pairwise_interactions": False,
    "holomorphic_operations_only": True,
    "dtype": "complex128/float64",
}
TRAIN_CONFIG = {
    "steps": TRAIN_STEPS,
    "batch_bar": 3416,
    "batch_boundary": 427,
    "batch_mono": 427,
    "optimizer": "Adam",
    "lr_initial": 5e-5,
    "lr_switch_step": 12_000,
    "lr_final": 1e-5,
    "preserve_adam_state_at_lr_switch": True,
    "grad_clip": 1.0,
    "log_every": 100,
    "sampler_domain": {"laplace_real": [-0.5, 5.0], "imag": [-5.0, 5.0]},
}
LOSS_WEIGHTS = {
    "lambda_bar": 1.0,
    "lambda_bdary": 1.0,
    "lambda_zero": 0.0,
    "lambda_mono": 1.0,
    "lambda_imag": 1.0,
}
ADAPTIVE_BAR = {
    "adaptive_bar_weight": True,
    "bar_weight_alpha": 0.3,
    "bar_weight_eps": 1e-12,
    "bar_weight_min": 0.5,
    "bar_weight_max": 2.0,
}

# This matches the sampled-constraint budget of the original 25k HCN
# protocol; it is not a matched-compute claim against NN4MGF.
ORIGINAL_HCN_PROTOCOL = {
    "steps": 25_000,
    "batch_bar": 2048,
    "batch_boundary": 256,
    "batch_mono": 256,
}


def sample_budget(protocol):
    bar = protocol["steps"] * protocol["batch_bar"]
    boundary = protocol["steps"] * protocol["batch_boundary"]
    mono = protocol["steps"] * protocol["batch_mono"]
    return {"bar": bar, "boundary": boundary, "mono": mono, "total": bar + boundary + mono}


PAPER_PROTOCOL = {
    "steps": TRAIN_STEPS,
    "batch_bar": TRAIN_CONFIG["batch_bar"],
    "batch_boundary": TRAIN_CONFIG["batch_boundary"],
    "batch_mono": TRAIN_CONFIG["batch_mono"],
}
ORIGINAL_BUDGET = sample_budget(ORIGINAL_HCN_PROTOCOL)
PAPER_BUDGET = sample_budget(PAPER_PROTOCOL)
BUDGET_DIFFERENCE_PERCENT = 100 * (
    PAPER_BUDGET["total"] - ORIGINAL_BUDGET["total"]
) / ORIGINAL_BUDGET["total"]
assert ORIGINAL_BUDGET["total"] == 64_000_000
assert PAPER_BUDGET["total"] == 64_050_000
assert abs(BUDGET_DIFFERENCE_PERCENT) <= 0.1
assert RUN_MODE in {"smoke", "train", "evaluate"}

OFFICIAL_PROVENANCE = {
    "method": "NN4MGF",
    "paper": "Deep Learning Method for Stationary Distribution of Reflected Brownian Motion",
    "repository": "https://github.com/zhangz73/NN4MGF",
    "commit": "1956666902281f6ce4558ca82d8d15fa038f06fb",
    "benchmark_script": "test_2d_harrison.py",
    "public_table": "Plots/d=2/harrison/tables.txt",
    "benchmark_script_sha256": "77ec5d19a3c7ae40b51bb55db6c659b63a91e4eb29ab151faad1c2e905f74648",
}

set_all_seeds(SEED)
print({
    "experiment": EXPERIMENT,
    "run_mode": RUN_MODE,
    "fixed_endpoint": TRAIN_STEPS,
    "talbot_degree": PRIMARY_TALBOT_DEGREE,
    "sampled_constraints": PAPER_BUDGET["total"],
    "budget_difference_percent": BUDGET_DIFFERENCE_PERCENT,
})


## 3. Harrison RBM benchmark and BAR coefficients


In [ ]:
@dataclass
class RBMParams:
    mu: torch.Tensor
    Sigma: torch.Tensor
    R: torch.Tensor
    alpha: torch.Tensor | None = None


def build_params():
    return RBMParams(
        mu=torch.tensor([-1.0, 0.0], dtype=r_dtype, device=device),
        Sigma=torch.eye(2, dtype=r_dtype, device=device),
        R=torch.tensor([[1.0, 0.0], [-1.0, 1.0]], dtype=r_dtype, device=device),
        alpha=None,
    )


def compute_boundary_mass(params):
    mass = -torch.linalg.solve(params.R, params.mu)
    if not torch.isfinite(mass).all() or torch.any(mass <= 0):
        raise ValueError("Boundary masses must be finite and positive.")
    return mass


def compute_gammas(theta, params):
    theta = theta.to(dtype=c_dtype, device=device)
    mu = params.mu.to(c_dtype)
    Sigma = params.Sigma.to(c_dtype)
    R = params.R.to(c_dtype)
    gamma0 = -0.5 * torch.sum((theta @ Sigma) * theta, dim=1, keepdim=True) + theta @ mu[:, None]
    gamma_all = -(theta @ R)
    return gamma0, gamma_all


def rbm_payload(params):
    return {
        "mu": params.mu.cpu().tolist(),
        "Sigma": params.Sigma.cpu().tolist(),
        "R": params.R.cpu().tolist(),
        "alpha": None,
    }


PARAMS = build_params()
assert torch.allclose(compute_boundary_mass(PARAMS), torch.tensor([1.0, 1.0], device=device))
print(rbm_payload(PARAMS))


## 4. Genuine non-product Holomorphic Cauchy Network

The interior log-transform is the sum of a centered shared-coordinate branch
and a centered full-input branch. The global branch receives
\((\theta_1,\theta_2)\) jointly, so its mixed interaction residual need not
vanish. All transform paths use complex linear maps and the rational Cauchy
activation. Hard origin and RBM boundary-mass anchors are retained.


In [ ]:
class CauchyComplex(nn.Module):
    def __init__(self, num_features, init_lambda=0.1, init_d=0.5, d_min=1e-3):
        super().__init__()
        self.num_features = int(num_features)
        self.d_min = float(d_min)
        self.lambda1 = nn.Parameter(torch.ones(num_features, dtype=c_dtype) * complex(init_lambda, 0.0))
        self.lambda2 = nn.Parameter(torch.ones(num_features, dtype=c_dtype) * complex(init_lambda, 0.0))
        init_value = max(init_d - d_min, 1e-6)
        init_rho = torch.log(torch.expm1(torch.tensor(init_value, dtype=r_dtype)))
        self.rho = nn.Parameter(torch.ones(num_features, dtype=r_dtype) * init_rho)
        self.last_min_abs_denominator = None
        self.running_min_abs_denominator = None

    def positive_d(self):
        return F.softplus(self.rho) + self.d_min

    def reset_denominator_tracking(self):
        self.last_min_abs_denominator = None
        self.running_min_abs_denominator = None

    def _update_denominator_tracking(self, denominator):
        current = torch.min(torch.abs(denominator)).detach()
        self.last_min_abs_denominator = current
        self.running_min_abs_denominator = current if self.running_min_abs_denominator is None else torch.minimum(self.running_min_abs_denominator, current).detach()

    def _denominator(self, z):
        d_complex = self.positive_d().to(device=z.device, dtype=z.dtype)
        return z ** 2 + d_complex ** 2, d_complex

    def forward(self, z):
        denominator, _ = self._denominator(z)
        self._update_denominator_tracking(denominator)
        output = (self.lambda1 * z + self.lambda2) / denominator
        if not torch.isfinite(output.real).all() or not torch.isfinite(output.imag).all():
            raise FloatingPointError("Non-finite Cauchy activation output.")
        return output

    def derivative(self, z):
        denominator, d_complex = self._denominator(z)
        self._update_denominator_tracking(denominator)
        output = (self.lambda1 * (d_complex ** 2 - z ** 2) - 2.0 * self.lambda2 * z) / (denominator ** 2)
        if not torch.isfinite(output.real).all() or not torch.isfinite(output.imag).all():
            raise FloatingPointError("Non-finite Cauchy derivative output.")
        return output


def reset_all_cauchy_denominator_tracking(model):
    for module in model.modules():
        if isinstance(module, CauchyComplex):
            module.reset_denominator_tracking()


def minimum_tracked_cauchy_denominator(model):
    values = [module.running_min_abs_denominator for module in model.modules() if isinstance(module, CauchyComplex) and module.running_min_abs_denominator is not None]
    if not values:
        return math.inf
    return float(torch.min(torch.stack(values)).detach().cpu())


class HolomorphicCauchyScalarNet(nn.Module):
    """Complex Linear -> Cauchy -> Complex Linear scalar map."""
    def __init__(self, input_dim, hidden_width):
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_width = int(hidden_width)
        self.linear_0 = nn.Linear(input_dim, hidden_width, dtype=c_dtype)
        self.cauchy_0 = CauchyComplex(hidden_width)
        self.linear_1 = nn.Linear(hidden_width, 1, dtype=c_dtype)

    def forward(self, x):
        if x.ndim != 2 or x.shape[1] != self.input_dim:
            raise ValueError(f"x must have shape [N,{self.input_dim}], got {tuple(x.shape)}")
        x = x.to(dtype=c_dtype, device=device)
        return self.linear_1(self.cauchy_0(self.linear_0(x)))

    def forward_with_input_jacobian(self, x):
        x = x.to(dtype=c_dtype, device=device)
        z = self.linear_0(x)
        activation = self.cauchy_0(z)
        output = self.linear_1(activation)
        derivative = self.cauchy_0.derivative(z)
        jacobian = torch.einsum("h,nh,hj->nj", self.linear_1.weight[0], derivative, self.linear_0.weight)
        return output, jacobian

    def forward_with_first_input_derivative(self, x):
        output, jacobian = self.forward_with_input_jacobian(x)
        return output, jacobian[:, 0:1]


def run_scalar_net_chunked(net, x, pair_chunk_size=32768, derivative_mode=None):
    outputs, derivatives = [], []
    for start in range(0, x.shape[0], pair_chunk_size):
        chunk = x[start:start + pair_chunk_size]
        if derivative_mode == "first":
            output, derivative = net.forward_with_first_input_derivative(chunk)
            derivatives.append(derivative)
        elif derivative_mode == "all":
            output, derivative = net.forward_with_input_jacobian(chunk)
            derivatives.append(derivative)
        else:
            output = net(chunk)
        outputs.append(output)
    output = torch.cat(outputs, dim=0)
    return (output, torch.cat(derivatives, dim=0)) if derivative_mode is not None else output


class StructuredNonproductGlobalCauchyMGF(nn.Module):
    """Coordinate branch + full-input global branch with structural origin anchor."""
    def __init__(self, d, coord_embed_dim, boundary_embed_dim, interior_hidden_width, boundary_hidden_width, global_hidden_width, pair_chunk_size, boundary_mass):
        super().__init__()
        self.d = int(d)
        self.coord_embed_dim = int(coord_embed_dim)
        self.boundary_embed_dim = int(boundary_embed_dim)
        self.pair_chunk_size = int(pair_chunk_size)
        boundary_mass = boundary_mass.detach().clone().to(dtype=r_dtype, device=device)
        if boundary_mass.shape != (self.d,) or not torch.isfinite(boundary_mass).all() or torch.any(boundary_mass <= 0):
            raise ValueError("boundary_mass must be finite, positive, and have shape [d].")
        self.register_buffer("boundary_mass", boundary_mass, persistent=True)
        self.register_buffer("boundary_log_mass", torch.log(boundary_mass).to(c_dtype), persistent=True)
        scale = 0.01
        self.coord_embedding = nn.Parameter(scale * torch.complex(torch.randn(self.d, self.coord_embed_dim, dtype=r_dtype), torch.randn(self.d, self.coord_embed_dim, dtype=r_dtype)))
        self.boundary_embedding = nn.Parameter(scale * torch.complex(torch.randn(self.d, self.boundary_embed_dim, dtype=r_dtype), torch.randn(self.d, self.boundary_embed_dim, dtype=r_dtype)))
        self.interior_net = HolomorphicCauchyScalarNet(1 + self.coord_embed_dim, interior_hidden_width)
        self.boundary_net = HolomorphicCauchyScalarNet(1 + self.coord_embed_dim + self.boundary_embed_dim, boundary_hidden_width)
        self.global_net = HolomorphicCauchyScalarNet(self.d, global_hidden_width)

    def _interior_inputs(self, theta):
        batch, d = theta.shape
        embedding = self.coord_embedding.view(1, d, self.coord_embed_dim).expand(batch, d, self.coord_embed_dim)
        return torch.cat([theta.unsqueeze(-1), embedding], dim=2).reshape(batch * d, 1 + self.coord_embed_dim)

    def _interior_f0_and_jac(self, theta, need_jacobian):
        batch, d = theta.shape
        inputs = self._interior_inputs(theta)
        if need_jacobian:
            coordinate_values, coordinate_derivatives = run_scalar_net_chunked(self.interior_net, inputs, self.pair_chunk_size, derivative_mode="first")
            coordinate_derivatives = coordinate_derivatives.view(batch, d)
            global_values, global_jacobian = run_scalar_net_chunked(self.global_net, theta, self.pair_chunk_size, derivative_mode="all")
        else:
            coordinate_values = run_scalar_net_chunked(self.interior_net, inputs, self.pair_chunk_size)
            global_values = run_scalar_net_chunked(self.global_net, theta, self.pair_chunk_size)
            coordinate_derivatives = global_jacobian = None
        coordinate_values = coordinate_values.view(batch, d)
        zero_coordinate_inputs = torch.cat([torch.zeros(d, 1, dtype=c_dtype, device=theta.device), self.coord_embedding], dim=1)
        coordinate_zero = run_scalar_net_chunked(self.interior_net, zero_coordinate_inputs, self.pair_chunk_size).view(1, d)
        global_zero = run_scalar_net_chunked(self.global_net, torch.zeros(1, d, dtype=c_dtype, device=theta.device), self.pair_chunk_size)
        f_coordinate = torch.sum(coordinate_values - coordinate_zero, dim=1, keepdim=True)
        f_global = global_values - global_zero
        return f_coordinate + f_global, None if not need_jacobian else coordinate_derivatives + global_jacobian

    def _boundary_inputs_for_k(self, theta, k):
        batch, d = theta.shape
        mask = torch.ones(d, dtype=torch.bool, device=theta.device)
        mask[k] = False
        selected = theta[:, mask]
        coordinate = self.coord_embedding[mask].view(1, d - 1, self.coord_embed_dim).expand(batch, d - 1, self.coord_embed_dim)
        boundary = self.boundary_embedding[k].view(1, 1, self.boundary_embed_dim).expand(batch, d - 1, self.boundary_embed_dim)
        return torch.cat([selected.unsqueeze(-1), coordinate, boundary], dim=2).reshape(batch * (d - 1), 1 + self.coord_embed_dim + self.boundary_embed_dim), mask

    def _boundary_zero_inputs_for_k(self, theta, k):
        mask = torch.ones(self.d, dtype=torch.bool, device=theta.device)
        mask[k] = False
        zeros = torch.zeros(self.d - 1, 1, dtype=c_dtype, device=theta.device)
        coordinate = self.coord_embedding[mask]
        boundary = self.boundary_embedding[k].view(1, self.boundary_embed_dim).expand(self.d - 1, self.boundary_embed_dim)
        return torch.cat([zeros, coordinate, boundary], dim=1)

    def _boundary_fk_and_deriv_for_k(self, theta, k, need_derivative):
        batch, d = theta.shape
        inputs, mask = self._boundary_inputs_for_k(theta, k)
        zero_inputs = self._boundary_zero_inputs_for_k(theta, k)
        if need_derivative:
            values, derivatives = run_scalar_net_chunked(self.boundary_net, inputs, self.pair_chunk_size, derivative_mode="first")
            derivatives = derivatives.view(batch, d - 1)
        else:
            values = run_scalar_net_chunked(self.boundary_net, inputs, self.pair_chunk_size)
            derivatives = None
        zero_values = run_scalar_net_chunked(self.boundary_net, zero_inputs, self.pair_chunk_size).view(1, d - 1)
        values = values.view(batch, d - 1)
        fk = self.boundary_log_mass[k].view(1, 1) + torch.sum(values - zero_values, dim=1, keepdim=True)
        return fk, derivatives, mask

    def forward_selected_boundary(self, theta, k):
        theta = theta.to(dtype=c_dtype, device=device)
        f0, _ = self._interior_f0_and_jac(theta, need_jacobian=False)
        fk, _, _ = self._boundary_fk_and_deriv_for_k(theta, k, need_derivative=False)
        return f0, fk

    def forward(self, theta):
        values, _ = self.forward_with_input_jacobian(theta, need_jacobian=False)
        return values

    def forward_with_input_jacobian(self, theta, need_jacobian=True):
        theta = theta.to(dtype=c_dtype, device=device)
        if theta.ndim != 2 or theta.shape[1] != self.d:
            raise ValueError(f"theta must have shape [B,{self.d}], got {tuple(theta.shape)}")
        batch, d = theta.shape
        f0, joint_jacobian = self._interior_f0_and_jac(theta, need_jacobian)
        boundary_columns = []
        jacobian = torch.zeros(batch, d + 1, d, dtype=c_dtype, device=theta.device) if need_jacobian else None
        if need_jacobian:
            jacobian[:, 0, :] = joint_jacobian
        for k in range(d):
            fk, derivatives, mask = self._boundary_fk_and_deriv_for_k(theta, k, need_jacobian)
            boundary_columns.append(fk)
            if need_jacobian:
                jacobian[:, k + 1, mask] = derivatives
        return torch.cat([f0] + boundary_columns, dim=1), jacobian


def build_model(params, seed, model_config=None):
    set_all_seeds(seed)
    cfg = dict(MODEL_CONFIG if model_config is None else model_config)
    if not cfg.get("full_input_global_branch") or not cfg.get("nonproduct_global_branch"):
        raise ValueError("This experiment requires the full-input non-product global branch.")
    return StructuredNonproductGlobalCauchyMGF(
        d=cfg["d"], coord_embed_dim=cfg["coord_embed_dim"], boundary_embed_dim=cfg["boundary_embed_dim"],
        interior_hidden_width=cfg["interior_hidden_width"], boundary_hidden_width=cfg["boundary_hidden_width"],
        global_hidden_width=cfg["global_hidden_width"], pair_chunk_size=cfg["pair_chunk_size"],
        boundary_mass=compute_boundary_mass(params),
    ).to(device)


def parameter_counts(model):
    tensor_elements = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    real_scalar_dof = sum((2 if parameter.is_complex() else 1) * parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return {"trainable_tensor_elements": int(tensor_elements), "trainable_real_scalar_dof": int(real_scalar_dof)}


def parameter_budget_structure(model):
    groups = {}
    for name, parameter in model.named_parameters():
        group = name.split(".", 1)[0]
        row = groups.setdefault(group, {"tensor_parameter_count": 0, "real_scalar_dof": 0, "parameter_names": []})
        row["tensor_parameter_count"] += int(parameter.numel())
        row["real_scalar_dof"] += int((2 if parameter.is_complex() else 1) * parameter.numel())
        row["parameter_names"].append(name)
    counts = parameter_counts(model)
    difference = counts["trainable_real_scalar_dof"] - TARGET_TRAINABLE_REAL_SCALAR_DOF
    return {
        "target_trainable_real_scalar_dof": TARGET_TRAINABLE_REAL_SCALAR_DOF,
        "module_breakdown": groups,
        **counts,
        "absolute_difference_from_target": abs(int(difference)),
        "signed_difference_from_target": int(difference),
        "percentage_difference_from_target": 100.0 * difference / TARGET_TRAINABLE_REAL_SCALAR_DOF,
        "percentage_of_target": 100.0 * counts["trainable_real_scalar_dof"] / TARGET_TRAINABLE_REAL_SCALAR_DOF,
        "within_95_to_100_percent_budget": 0.95 * TARGET_TRAINABLE_REAL_SCALAR_DOF <= counts["trainable_real_scalar_dof"] <= TARGET_TRAINABLE_REAL_SCALAR_DOF,
        "dummy_parameters": False,
        "frozen_parameters_used_to_adjust_count": False,
        "all_parameters_participate_in_forward": None,
        "coordinate_branch_nonzero_gradient": None,
        "global_branch_nonzero_gradient": None,
    }


MODEL_FOR_AUDIT = build_model(PARAMS, SEED)
PARAMETER_COUNTS = parameter_counts(MODEL_FOR_AUDIT)
PARAMETER_BUDGET_AUDIT = parameter_budget_structure(MODEL_FOR_AUDIT)
assert PARAMETER_COUNTS == {"trainable_tensor_elements": 7784, "trainable_real_scalar_dof": 15236}, PARAMETER_COUNTS
assert PARAMETER_BUDGET_AUDIT["within_95_to_100_percent_budget"]
print(json.dumps(PARAMETER_BUDGET_AUDIT, indent=2))
del MODEL_FOR_AUDIT


## 5. Complex-domain sampler


In [ ]:
def sample_complex_box(N, d=2, re_min=-0.5, re_max=5.0, im_max=5.0, requires_grad=False, device=device):
    real = re_min + (re_max - re_min) * torch.rand(N, d, dtype=r_dtype, device=device)
    imag = -im_max + 2.0 * im_max * torch.rand(N, d, dtype=r_dtype, device=device)
    theta = torch.complex(real, imag)
    if requires_grad:
        theta.requires_grad_(True)
    return theta


def sample_nn4mgf_vector_mgf(N, d=2, lb=-5.0, ub=0.5, imag_lb=-5.0, imag_ub=5.0, device=device):
    real_lb_curr = ub - (ub - lb) * torch.rand(N, 1, dtype=r_dtype, device=device)
    imag_bd = max(abs(imag_ub), abs(imag_lb)) * torch.rand(N, 1, dtype=r_dtype, device=device)
    real = (ub - real_lb_curr) * torch.rand(N, d, dtype=r_dtype, device=device) + real_lb_curr
    imag = 2.0 * imag_bd * torch.rand(N, d, dtype=r_dtype, device=device) - imag_bd
    return torch.complex(real, imag)


def sample_nn4mgf_vector_laplace(N, d=2, lap_re_min=-0.5, lap_re_max=5.0, im_max=5.0, requires_grad=False, device=device):
    theta = -sample_nn4mgf_vector_mgf(N=N, d=d, lb=-lap_re_max, ub=-lap_re_min, imag_lb=-im_max, imag_ub=im_max, device=device)
    if requires_grad:
        theta.requires_grad_(True)
    return theta


def sample_training_theta(N, d=2, re_min=-0.5, re_max=5.0, im_max=5.0, requires_grad=False, device=device):
    return sample_nn4mgf_vector_laplace(N=N, d=d, lap_re_min=re_min, lap_re_max=re_max, im_max=im_max, requires_grad=requires_grad, device=device)


def sample_nested_real_radius(N, lb=0.0, ub=5.0, device=device):
    # Near-zero-biased sampler: E[s] = lb + (ub-lb)/3 for U(0,1)^2.
    u = torch.rand(N, 1, dtype=r_dtype, device=device)
    return lb + (ub - lb) * (u ** 2)


def sample_mixed_bar_theta(N=4000, d=2, re_min=-0.5, re_max=5.0, im_max=5.0, device=device):
    """
    Mixed 4000-point BAR sampler proportions:
      2000 global, 800 exact complex diagonal, 800 near-diagonal complex, 400 positive-real diagonal.
    Counts are scaled proportionally for N != 4000.
    """
    n_global = int(round(N * 0.50))
    n_exact_diag = int(round(N * 0.20))
    n_near_diag = int(round(N * 0.20))
    n_pos_diag = N - n_global - n_exact_diag - n_near_diag

    global_theta = sample_training_theta(n_global, d=d, re_min=re_min, re_max=re_max, im_max=im_max, device=device)
    s_complex = sample_training_theta(n_exact_diag, d=1, re_min=re_min, re_max=re_max, im_max=im_max, device=device)
    exact_diag = s_complex.expand(n_exact_diag, d)

    s_near = sample_training_theta(n_near_diag, d=1, re_min=re_min, re_max=re_max, im_max=im_max, device=device)
    noise = 0.05 * sample_complex_box(n_near_diag, d=d, re_min=-1.0, re_max=1.0, im_max=1.0, device=device)
    near_diag_raw = s_near.expand(n_near_diag, d) + noise
    near_diag = torch.complex(
        torch.clamp(near_diag_raw.real, min=re_min, max=re_max),
        torch.clamp(near_diag_raw.imag, min=-im_max, max=im_max),
    )

    positive_lb = max(0.0, re_min)
    positive_ub = re_max
    if positive_ub <= positive_lb:
        raise ValueError(f"Positive-real diagonal domain is empty: [{positive_lb}, {positive_ub}]")
    s_real = sample_nested_real_radius(n_pos_diag, lb=positive_lb, ub=positive_ub, device=device)
    pos_diag = torch.complex(s_real.expand(n_pos_diag, d), torch.zeros(n_pos_diag, d, dtype=r_dtype, device=device))

    theta = torch.cat([global_theta, exact_diag, near_diag, pos_diag], dim=0).to(dtype=c_dtype, device=device)
    return theta[torch.randperm(theta.shape[0], device=device)]

## 6. Unsupervised BAR objective


In [ ]:
def construct_boundary_directions(params):
    R_c = params.R.to(dtype=c_dtype, device=device)
    A = R_c.T
    d = R_c.shape[0]
    eye = torch.eye(d, dtype=c_dtype, device=device)
    rows = []
    for k in range(d):
        M = A.clone()
        M[k, :] = eye[k, :]
        rows.append(torch.linalg.solve(M, eye[k, :]).view(1, d))
    return torch.cat(rows, dim=0)


def construct_boundary_thetas(theta, params):
    theta = theta.to(dtype=c_dtype, device=device)
    _, d = theta.shape
    V = construct_boundary_directions(params)
    return [theta[:, k:k+1] * V[k:k+1, :] for k in range(d)], V


# ============================================================
# 8-10. BAR, structured boundary loss, fixed held-out evaluation
# ============================================================

DEFAULT_ADAPTIVE_BAR_CONFIG = {
    "adaptive_bar_weight": True,
    "bar_weight_alpha": 0.3,
    "bar_weight_eps": 1e-12,
    "bar_weight_min": 0.5,
    "bar_weight_max": 2.0,
}


def with_adaptive_bar_defaults(config):
    merged = dict(DEFAULT_ADAPTIVE_BAR_CONFIG)
    merged.update(config)
    return merged


def adaptive_bar_kwargs(config):
    cfg = with_adaptive_bar_defaults(config)
    return {
        "adaptive_bar_weight": bool(cfg["adaptive_bar_weight"]),
        "bar_weight_alpha": float(cfg["bar_weight_alpha"]),
        "bar_weight_eps": float(cfg["bar_weight_eps"]),
        "bar_weight_min": float(cfg["bar_weight_min"]),
        "bar_weight_max": float(cfg["bar_weight_max"]),
    }


def split_f_tuple(f_out):
    if isinstance(f_out, torch.Tensor):
        return f_out
    if isinstance(f_out, (tuple, list)):
        return torch.cat(list(f_out), dim=1)
    raise TypeError(f"Unsupported model output type {type(f_out)!r}")


def compute_bar_pointwise(theta, f_tuple, params, eps_gamma=1e-12, eps_bar=1e-8):
    f_mat = split_f_tuple(f_tuple)
    gamma0, gamma_all = compute_gammas(theta, params)
    gamma_mat = torch.cat([gamma0, gamma_all], dim=1)
    nu = torch.max(torch.log(torch.abs(gamma_mat) + eps_gamma) + f_mat.real, dim=1, keepdim=True)[0]
    kappa = gamma_mat * torch.exp(f_mat - nu.to(dtype=c_dtype))
    left = kappa[:, 0:1]
    right = torch.sum(kappa[:, 1:], dim=1, keepdim=True)
    return (torch.abs(left - right) / (torch.abs(left) + torch.abs(right) + eps_bar)) ** 2


def weighted_normalized_mean(pointwise, adaptive_bar_weight=True, bar_weight_alpha=0.3, bar_weight_eps=1e-12, bar_weight_min=0.5, bar_weight_max=2.0):
    unweighted = torch.mean(pointwise)
    if adaptive_bar_weight:
        raw_w = (pointwise.detach() + bar_weight_eps) ** bar_weight_alpha
        weights = torch.clamp(raw_w / (torch.mean(raw_w) + bar_weight_eps), min=bar_weight_min, max=bar_weight_max).detach()
        weighted = torch.sum(weights * pointwise) / (torch.sum(weights) + bar_weight_eps)
    else:
        weights = torch.ones_like(pointwise).detach()
        weighted = unweighted
    return weighted, unweighted, weights


def compute_bar_loss(theta, f_tuple, params, eps_gamma=1e-12, eps_bar=1e-8, adaptive_bar_weight=True, bar_weight_alpha=0.3, bar_weight_eps=1e-12, bar_weight_min=0.5, bar_weight_max=2.0, return_diagnostics=False):
    pointwise = compute_bar_pointwise(theta, f_tuple, params, eps_gamma=eps_gamma, eps_bar=eps_bar)
    weighted, unweighted, weights = weighted_normalized_mean(
        pointwise,
        adaptive_bar_weight=adaptive_bar_weight,
        bar_weight_alpha=bar_weight_alpha,
        bar_weight_eps=bar_weight_eps,
        bar_weight_min=bar_weight_min,
        bar_weight_max=bar_weight_max,
    )
    if not return_diagnostics:
        return weighted
    return weighted, {
        "bar_unweighted": unweighted.detach(),
        "bar_weighted": weighted.detach(),
        "bar_weight_mean": torch.mean(weights).detach(),
        "bar_weight_min": torch.min(weights).detach(),
        "bar_weight_max": torch.max(weights).detach(),
    }


def boundary_face_residual_from_f(theta_k, f0, fk, params, k, eps_gamma=1e-12, eps_bar=1e-8):
    gamma0, gamma_all = compute_gammas(theta_k, params)
    gammak = gamma_all[:, k:k+1]
    nu = torch.maximum(torch.log(torch.abs(gamma0) + eps_gamma) + f0.real, torch.log(torch.abs(gammak) + eps_gamma) + fk.real).to(dtype=c_dtype)
    left = gamma0 * torch.exp(f0 - nu)
    right = gammak * torch.exp(fk - nu)
    rel = torch.abs(left - right) / (torch.abs(left) + torch.abs(right) + eps_bar)
    return rel ** 2


def compute_boundary_multiplicative_loss_generic(model, theta, params, eps_gamma=1e-12, eps_bar=1e-8):
    theta = theta.to(dtype=c_dtype, device=device)
    _, d = theta.shape
    tilde_thetas, _ = construct_boundary_thetas(theta, params)
    total = torch.zeros((), dtype=r_dtype, device=device)
    for k, theta_k in enumerate(tilde_thetas):
        f_mat = split_f_tuple(model(theta_k))
        total = total + torch.mean(boundary_face_residual_from_f(theta_k, f_mat[:, 0:1], f_mat[:, k+1:k+2], params, k, eps_gamma, eps_bar))
    return total / d


def compute_boundary_multiplicative_loss_structured(model, theta, params, eps_gamma=1e-12, eps_bar=1e-8):
    theta = theta.to(dtype=c_dtype, device=device)
    _, d = theta.shape
    tilde_thetas, _ = construct_boundary_thetas(theta, params)
    total = torch.zeros((), dtype=r_dtype, device=device)
    for k, theta_k in enumerate(tilde_thetas):
        f0, fk = model.forward_selected_boundary(theta_k, k)
        total = total + torch.mean(boundary_face_residual_from_f(theta_k, f0, fk, params, k, eps_gamma, eps_bar))
    return total / d


def evaluate_boundary_residual_independent(model, theta_bdary_eval, params, eps_gamma=1e-12, eps_bar=1e-8):
    theta = theta_bdary_eval.to(dtype=c_dtype, device=device)
    _, d = theta.shape
    tilde_thetas, _ = construct_boundary_thetas(theta, params)
    face_means = []
    for k, theta_k in enumerate(tilde_thetas):
        f0, fk = model.forward_selected_boundary(theta_k, k)
        face_means.append(torch.mean(boundary_face_residual_from_f(theta_k, f0, fk, params, k, eps_gamma, eps_bar)))
    face_tensor = torch.stack(face_means)
    return {
        "eval_boundary_mean": torch.mean(face_tensor),
        "eval_boundary_max": torch.max(face_tensor),
        "eval_boundary_worst_face": int(torch.argmax(face_tensor).detach().cpu()),
        "eval_boundary_per_face": face_tensor.detach(),
    }


def compute_zero_loss(model, params):
    theta0 = torch.zeros(1, params.R.shape[0], dtype=c_dtype, device=device)
    return torch.mean(torch.abs(torch.exp(model(theta0)[:, 0:1]) - 1.0) ** 2)


def compute_monotonicity_components_analytical_from_theta_real(model, theta_real):
    theta = torch.complex(theta_real.to(dtype=r_dtype, device=device), torch.zeros_like(theta_real.to(dtype=r_dtype, device=device)))
    f_mat, J = model.forward_with_input_jacobian(theta, need_jacobian=True)
    loss_mono = torch.mean(torch.relu(J.real) ** 2)
    loss_imag = torch.mean(f_mat.imag ** 2)
    return loss_mono, loss_imag


def compute_monotonicity_components_analytical(model, N_mono, d=2, re_min=-0.5, re_max=5.0):
    theta = sample_training_theta(N_mono, d=d, re_min=re_min, re_max=re_max, im_max=0.0, device=device)
    return compute_monotonicity_components_analytical_from_theta_real(model, theta.real.detach())


def compute_full_training_loss_structured(model, params, batch_bar=4000, batch_bdary=256, batch_mono=256, d=2, lambda_bar=1.0, lambda_bdary=1.0, lambda_zero=0.0, lambda_mono=1.0, lambda_imag=1.0, mono_re_min=-0.5, mono_re_max=5.0, adaptive_bar_weight=True, bar_weight_alpha=0.3, bar_weight_eps=1e-12, bar_weight_min=0.5, bar_weight_max=2.0, return_diagnostics=False):
    theta_bar = sample_mixed_bar_theta(N=batch_bar, d=d, re_min=-0.5, re_max=5.0, im_max=5.0, device=device)
    loss_bar, bar_diag = compute_bar_loss(
        theta_bar,
        model(theta_bar),
        params,
        adaptive_bar_weight=adaptive_bar_weight,
        bar_weight_alpha=bar_weight_alpha,
        bar_weight_eps=bar_weight_eps,
        bar_weight_min=bar_weight_min,
        bar_weight_max=bar_weight_max,
        return_diagnostics=True,
    )
    theta_bdary = sample_training_theta(batch_bdary, d=d, re_min=-0.5, re_max=5.0, im_max=5.0, device=device)
    loss_bdary = compute_boundary_multiplicative_loss_structured(model, theta_bdary, params)
    if lambda_zero == 0.0:
        loss_zero = torch.zeros((), dtype=r_dtype, device=device)
    else:
        loss_zero = compute_zero_loss(model, params)
    loss_mono, loss_imag = compute_monotonicity_components_analytical(model, batch_mono, d=d, re_min=mono_re_min, re_max=mono_re_max)
    total = lambda_bar*loss_bar + lambda_bdary*loss_bdary + lambda_zero*loss_zero + lambda_mono*loss_mono + lambda_imag*loss_imag
    if not return_diagnostics:
        return total
    diag = {"total_loss": total.detach(), "loss_bar": loss_bar.detach(), "loss_bar_unweighted": bar_diag["bar_unweighted"], "loss_bar_weighted": bar_diag["bar_weighted"], "loss_bdary": loss_bdary.detach(), "loss_zero": loss_zero.detach(), "loss_mono": loss_mono.detach(), "loss_imag": loss_imag.detach()}
    diag.update(bar_diag)
    return total, diag

## 7. Preflight, smoke test, and fixed-endpoint training

Preflight checks cover hard anchors, analytic derivatives, all parameter
gradients, coordinate/global branch gradients, Cauchy-pole distance, the exact
15,236-real-DOF budget, and a nonzero mixed interaction residual. Formal
training logs every 100 updates and saves only the prespecified step-15,000
checkpoint.


In [ ]:
LOG_COLUMNS = [
    "step", "lr", "total_loss", "loss_bar", "loss_bar_unweighted",
    "loss_boundary", "loss_mono", "loss_imag", "min_abs_cauchy_denominator",
    "cumulative_sampled_constraints", "wall_clock_seconds",
]

INTERACTION_TEST_POINTS = [
    (complex(0.3, 0.2), complex(-0.4, 0.5)),
    (complex(0.8, -0.35), complex(0.25, 0.6)),
    (complex(-0.2, 0.7), complex(1.1, -0.15)),
]


def check_gradients(model):
    for name, parameter in model.named_parameters():
        if parameter.grad is None:
            raise FloatingPointError(f"Missing gradient: {name}")
        grad = parameter.grad
        if not torch.isfinite(grad.real if grad.is_complex() else grad).all():
            raise FloatingPointError(f"Non-finite gradient: {name}")
        if grad.is_complex() and not torch.isfinite(grad.imag).all():
            raise FloatingPointError(f"Non-finite imaginary gradient: {name}")


def branch_gradient_norm(model, prefixes):
    total = 0.0
    for name, parameter in model.named_parameters():
        if any(name.startswith(prefix) for prefix in prefixes) and parameter.grad is not None:
            total += float(torch.sum(torch.abs(parameter.grad) ** 2).detach().cpu())
    return math.sqrt(total)


def interaction_residual_audit(model, label):
    was_training = model.training
    model.eval()
    rows = []
    with torch.no_grad():
        zero = torch.zeros(1, D, dtype=c_dtype, device=device)
        f00 = model(zero)[0, 0]
        for theta1, theta2 in INTERACTION_TEST_POINTS:
            both = torch.tensor([[theta1, theta2]], dtype=c_dtype, device=device)
            first = torch.tensor([[theta1, 0j]], dtype=c_dtype, device=device)
            second = torch.tensor([[0j, theta2]], dtype=c_dtype, device=device)
            residual = model(both)[0, 0] - model(first)[0, 0] - model(second)[0, 0] + f00
            value = complex(residual.detach().cpu())
            rows.append({
                "theta1": [theta1.real, theta1.imag],
                "theta2": [theta2.real, theta2.imag],
                "residual_real": value.real,
                "residual_imag": value.imag,
                "residual_abs": abs(value),
            })
    model.train(was_training)
    maximum = max(row["residual_abs"] for row in rows)
    return {
        "label": label,
        "test_points": rows,
        "maximum_interaction_residual_abs": maximum,
        "interaction_residual_passed": bool(math.isfinite(maximum) and maximum > 1e-12),
    }


def preflight_check():
    model = build_model(PARAMS, SEED)
    theta = sample_mixed_bar_theta(20, d=D)
    values, jacobian = model.forward_with_input_jacobian(theta, need_jacobian=True)
    assert values.shape == (20, D + 1)
    assert jacobian.shape == (20, D + 1, D)
    assert torch.isfinite(values.real).all() and torch.isfinite(values.imag).all()
    assert torch.isfinite(jacobian.real).all() and torch.isfinite(jacobian.imag).all()

    origin = model(torch.zeros(1, D, dtype=c_dtype, device=device))
    assert abs(complex(origin[0, 0].detach().cpu())) < 1e-12
    expected_boundary = torch.log(compute_boundary_mass(PARAMS)).to(c_dtype)
    assert torch.max(torch.abs(origin[0, 1:] - expected_boundary)).item() < 1e-12

    coordinate_probe = torch.complex(
        torch.randn(7, 1 + MODEL_CONFIG["coord_embed_dim"]),
        torch.randn(7, 1 + MODEL_CONFIG["coord_embed_dim"]),
    ).to(device)
    _, coordinate_analytic = model.interior_net.forward_with_first_input_derivative(coordinate_probe)
    coordinate_direction = torch.zeros_like(coordinate_probe)
    coordinate_direction[:, 0] = 1.0
    epsilon = 1e-6
    coordinate_finite_difference = (
        model.interior_net(coordinate_probe + epsilon * coordinate_direction)
        - model.interior_net(coordinate_probe - epsilon * coordinate_direction)
    ) / (2 * epsilon)
    coordinate_derivative_error = float(
        torch.max(torch.abs(coordinate_analytic - coordinate_finite_difference)).cpu()
    )
    assert coordinate_derivative_error < 2e-7

    global_probe = torch.tensor(
        [[0.2 + 0.3j, -0.4 + 0.1j], [0.7 - 0.2j, 0.15 + 0.5j]],
        dtype=c_dtype,
        device=device,
    )
    _, global_analytic = model.global_net.forward_with_input_jacobian(global_probe)
    global_columns = []
    for column in range(D):
        direction = torch.zeros_like(global_probe)
        direction[:, column] = 1.0
        global_columns.append(
            (model.global_net(global_probe + epsilon * direction)
             - model.global_net(global_probe - epsilon * direction)) / (2 * epsilon)
        )
    global_finite_difference = torch.cat(global_columns, dim=1)
    global_derivative_error = float(
        torch.max(torch.abs(global_analytic - global_finite_difference)).cpu()
    )
    assert global_derivative_error < 2e-7

    reset_all_cauchy_denominator_tracking(model)
    total, _ = compute_full_training_loss_structured(
        model, PARAMS, batch_bar=32, batch_bdary=8, batch_mono=8,
        d=D, return_diagnostics=True, **LOSS_WEIGHTS, **ADAPTIVE_BAR,
    )
    total.backward()
    check_gradients(model)
    coordinate_gradient = branch_gradient_norm(model, ("coord_embedding", "interior_net"))
    boundary_gradient = branch_gradient_norm(model, ("boundary_embedding", "boundary_net"))
    global_gradient = branch_gradient_norm(model, ("global_net",))
    assert coordinate_gradient > 0.0 and boundary_gradient > 0.0 and global_gradient > 0.0
    pole_distance = minimum_tracked_cauchy_denominator(model)
    assert math.isfinite(pole_distance) and pole_distance > POLE_TOL
    interaction = interaction_residual_audit(model, "initialization")
    assert interaction["interaction_residual_passed"]

    PARAMETER_BUDGET_AUDIT["all_parameters_participate_in_forward"] = True
    PARAMETER_BUDGET_AUDIT["coordinate_branch_nonzero_gradient"] = True
    PARAMETER_BUDGET_AUDIT["boundary_branch_nonzero_gradient"] = True
    PARAMETER_BUDGET_AUDIT["global_branch_nonzero_gradient"] = True
    PARAMETER_BUDGET_AUDIT["coordinate_branch_gradient_norm"] = coordinate_gradient
    PARAMETER_BUDGET_AUDIT["boundary_branch_gradient_norm"] = boundary_gradient
    PARAMETER_BUDGET_AUDIT["global_branch_gradient_norm"] = global_gradient
    return {
        "anchors": "ok",
        "coordinate_jacobian_max_abs_error": coordinate_derivative_error,
        "global_jacobian_max_abs_error": global_derivative_error,
        "gradients": "ok",
        "coordinate_branch_gradient_norm": coordinate_gradient,
        "boundary_branch_gradient_norm": boundary_gradient,
        "global_branch_gradient_norm": global_gradient,
        "minimum_abs_cauchy_denominator": pole_distance,
        "initial_nonseparability": interaction,
    }


def one_training_step(model, optimizer, batch_bar, batch_boundary, batch_mono):
    reset_all_cauchy_denominator_tracking(model)
    optimizer.zero_grad(set_to_none=True)
    total, diagnostics = compute_full_training_loss_structured(
        model, PARAMS,
        batch_bar=batch_bar, batch_bdary=batch_boundary, batch_mono=batch_mono,
        d=D, return_diagnostics=True, **LOSS_WEIGHTS, **ADAPTIVE_BAR,
    )
    if not torch.isfinite(total):
        raise FloatingPointError("Non-finite training loss.")
    pole_distance = minimum_tracked_cauchy_denominator(model)
    if not math.isfinite(pole_distance) or pole_distance <= POLE_TOL:
        raise FloatingPointError(f"Cauchy pole proximity failure: {pole_distance}")
    total.backward()
    check_gradients(model)
    torch.nn.utils.clip_grad_norm_(model.parameters(), TRAIN_CONFIG["grad_clip"])
    optimizer.step()
    return total, diagnostics, pole_distance


def environment_payload():
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "mpmath": mp.__version__,
        "cuda_version": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "device": str(device),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "real_dtype": str(r_dtype),
        "complex_dtype": str(c_dtype),
    }


def gpu_memory_payload():
    if not torch.cuda.is_available():
        return {"peak_gpu_allocated_bytes": 0, "peak_gpu_reserved_bytes": 0}
    return {
        "peak_gpu_allocated_bytes": int(torch.cuda.max_memory_allocated()),
        "peak_gpu_reserved_bytes": int(torch.cuda.max_memory_reserved()),
    }


def new_run_dir():
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    run_dir = OUTPUT_ROOT / f"{EXPERIMENT}_{stamp}"
    run_dir.mkdir(exist_ok=False)
    return run_dir.resolve()


def run_smoke_test():
    model = build_model(PARAMS, SEED)
    optimizer = torch.optim.Adam(model.parameters(), lr=TRAIN_CONFIG["lr_initial"])
    losses = []
    for _ in range(2):
        total, _, _ = one_training_step(model, optimizer, 16, 4, 4)
        losses.append(float(total.detach().cpu()))
    interaction = interaction_residual_audit(model, "after_2_step_smoke")
    assert interaction["interaction_residual_passed"]
    return {
        "steps": 2,
        "losses": losses,
        "nonseparability": interaction,
        "formal_outputs_created": False,
    }


def train_paper_protocol():
    run_dir = new_run_dir()
    run_config = {
        "experiment": EXPERIMENT,
        "seed": SEED,
        "fixed_primary_endpoint": TRAIN_STEPS,
        "model": MODEL_CONFIG,
        "training": TRAIN_CONFIG,
        "loss_weights": LOSS_WEIGHTS,
        "adaptive_bar": ADAPTIVE_BAR,
        "rbm": rbm_payload(PARAMS),
        "parameter_counts": PARAMETER_COUNTS,
        "sample_budget": PAPER_BUDGET,
        "original_hcn_sample_budget": ORIGINAL_BUDGET,
        "nonproduct_global_branch": True,
        "ground_truth_used_in_training": False,
        "ground_truth_used_for_checkpoint_selection": False,
        "checkpoint_selection": "none; the only saved model is the prespecified step-15000 endpoint",
        "evaluation_degree": PRIMARY_TALBOT_DEGREE,
        "raw_predictions": True,
        "prediction_clipping": False,
    }
    write_json(run_dir / "run_config.json", run_config)
    write_json(run_dir / "environment.json", environment_payload())
    write_json(run_dir / "sample_budget.json", {
        "paper_protocol": PAPER_PROTOCOL,
        "paper_budget": PAPER_BUDGET,
        "reference_25k_protocol": ORIGINAL_HCN_PROTOCOL,
        "reference_25k_budget": ORIGINAL_BUDGET,
        "relative_difference_percent": BUDGET_DIFFERENCE_PERCENT,
    })
    write_json(run_dir / "preflight_audit.json", PREFLIGHT_REPORT)
    write_json(run_dir / "parameter_budget_audit.json", PARAMETER_BUDGET_AUDIT)

    model = build_model(PARAMS, SEED)
    optimizer = torch.optim.Adam(model.parameters(), lr=TRAIN_CONFIG["lr_initial"])
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    cuda_sync()
    started = time.perf_counter()
    history = []

    for step in range(1, TRAIN_STEPS + 1):
        if step == TRAIN_CONFIG["lr_switch_step"] + 1:
            for group in optimizer.param_groups:
                group["lr"] = TRAIN_CONFIG["lr_final"]

        total, diagnostics, pole_distance = one_training_step(
            model,
            optimizer,
            TRAIN_CONFIG["batch_bar"],
            TRAIN_CONFIG["batch_boundary"],
            TRAIN_CONFIG["batch_mono"],
        )
        if step % TRAIN_CONFIG["log_every"] == 0 or step == TRAIN_STEPS:
            cuda_sync()
            sampled = step * (
                TRAIN_CONFIG["batch_bar"]
                + TRAIN_CONFIG["batch_boundary"]
                + TRAIN_CONFIG["batch_mono"]
            )
            row = {
                "step": step,
                "lr": optimizer.param_groups[0]["lr"],
                "total_loss": float(total.detach().cpu()),
                "loss_bar": float(diagnostics["loss_bar"].cpu()),
                "loss_bar_unweighted": float(diagnostics["loss_bar_unweighted"].cpu()),
                "loss_boundary": float(diagnostics["loss_bdary"].cpu()),
                "loss_mono": float(diagnostics["loss_mono"].cpu()),
                "loss_imag": float(diagnostics["loss_imag"].cpu()),
                "min_abs_cauchy_denominator": pole_distance,
                "cumulative_sampled_constraints": sampled,
                "wall_clock_seconds": time.perf_counter() - started,
            }
            history.append(row)
            write_csv(run_dir / "training_log.csv", history, LOG_COLUMNS)
            print(f"step={step:5d} loss={row['total_loss']:.4e} elapsed={row['wall_clock_seconds']:.1f}s")

    cuda_sync()
    training_elapsed = time.perf_counter() - started
    post_training_interaction = interaction_residual_audit(model, "post_training_step_15000")
    assert post_training_interaction["interaction_residual_passed"]
    nonseparability = {
        "architecture": "centered coordinate branch plus centered full-input global Holomorphic Cauchy branch",
        "initialization": PREFLIGHT_REPORT["initial_nonseparability"],
        "post_training": post_training_interaction,
    }
    write_json(run_dir / "nonseparability_audit.json", nonseparability)

    fig, axis = plt.subplots(figsize=(7.2, 4.6))
    axis.semilogy([row["step"] for row in history], [row["total_loss"] for row in history], label="total")
    axis.semilogy([row["step"] for row in history], [row["loss_bar"] for row in history], label="BAR")
    axis.semilogy([row["step"] for row in history], [row["loss_boundary"] for row in history], label="boundary")
    axis.axvline(TRAIN_CONFIG["lr_switch_step"], color="black", linestyle="--", linewidth=1, label="learning-rate switch")
    axis.set(xlabel="optimizer step", ylabel="loss", title="Fixed 15k training trajectory")
    axis.grid(True, alpha=0.3)
    axis.legend()
    fig.tight_layout()
    fig.savefig(run_dir / "training_loss.png", dpi=180)
    plt.close(fig)

    checkpoint_path = run_dir / "checkpoint_step15000.pt"
    torch.save({
        "experiment": EXPERIMENT,
        "seed": SEED,
        "step": TRAIN_STEPS,
        "model_config": MODEL_CONFIG,
        "train_config": TRAIN_CONFIG,
        "rbm": rbm_payload(PARAMS),
        "parameter_counts": PARAMETER_COUNTS,
        "nonseparability_audit": nonseparability,
        "model_state_dict": model.state_dict(),
    }, checkpoint_path)
    runtime = {
        "training_wall_clock_seconds": training_elapsed,
        "training_updates": TRAIN_STEPS,
        "sampled_constraints": PAPER_BUDGET,
        **gpu_memory_payload(),
    }
    write_json(run_dir / "training_runtime.json", runtime)
    write_json(run_dir / "checkpoint_sha256.json", {
        "checkpoint": checkpoint_path.name,
        "sha256": sha256_file(checkpoint_path),
    })
    return run_dir


PREFLIGHT_REPORT = preflight_check()
print("Preflight:", PREFLIGHT_REPORT)

FORMAL_RUN_DIR = None
if RUN_MODE == "smoke":
    print("Smoke test:", run_smoke_test())
elif RUN_MODE == "train":
    FORMAL_RUN_DIR = train_paper_protocol()
    print("Training complete:", FORMAL_RUN_DIR)


## 8. Post-training Degree-5 evaluation

Harrison ground truth and the published NN4MGF predictions first appear below,
after all training code. Evaluation loads only the fixed step-15,000 checkpoint,
uses Talbot Degree 5 at \(t=1,\ldots,5\), reports raw predictions without
clipping, rechecks nonseparability, and verifies that checkpoint bytes are
unchanged. The removable singularity of the tail transform at the origin is
filled by the model-implied mean obtained from its analytic origin Jacobian.
No reliable closed-form exact joint transform is available for an
independent Talbot inversion-floor calculation, so none is fabricated.


In [ ]:
# Ground truth and published predictions are introduced only after training.
mp.mp.dps = 50


def model_sum_laplace(model, s):
    z = complex(s)
    theta = torch.full((1, D), z, dtype=c_dtype, device=device)
    with torch.no_grad():
        value = torch.exp(model(theta)[0, 0]).detach().cpu().item()
    return mp.mpc(value.real, value.imag)


def model_tail_zero_limit(model):
    theta_zero = torch.zeros(1, D, dtype=c_dtype, device=device)
    with torch.no_grad():
        _, jacobian = model.forward_with_input_jacobian(
            theta_zero, need_jacobian=True
        )
    learned_mean = float((-torch.sum(jacobian[0, 0, :]).real).detach().cpu())
    if not math.isfinite(learned_mean) or learned_mean < 0.0:
        raise FloatingPointError(f"Invalid learned tail-transform limit: {learned_mean}")
    return mp.mpf(str(learned_mean))


def invert_tail(laplace_transform, t, zero_limit):
    def transformed_tail(s):
        if abs(s) < mp.mpf("1e-30"):
            return zero_limit
        return (1 - laplace_transform(s)) / s
    return mp.invertlaplace(
        transformed_tail,
        mp.mpf(str(t)),
        method="talbot",
        degree=PRIMARY_TALBOT_DEGREE,
    )


def parse_raw_prediction(value):
    real = float(mp.re(value))
    imag = float(mp.im(value))
    finite = math.isfinite(real) and math.isfinite(imag)
    invalid = (not finite) or abs(imag) > 1e-8 or real < 0.0 or real > 1.0
    return real, imag, invalid


def harrison_true_tail(t):
    # Polar rewrite of the official nested density integral over x1+x2>t.
    t = mp.mpf(str(t))
    constant = mp.pi ** (-mp.mpf("0.5")) * mp.mpf("2") ** mp.mpf("1.5")

    def tail_angle(angle):
        rate = 1 + mp.cos(angle)
        lower_radius = t / (mp.cos(angle) + mp.sin(angle))
        return (
            constant
            * mp.cos(angle / 2)
            * mp.gammainc(mp.mpf("1.5"), rate * lower_radius, mp.inf)
            / rate ** mp.mpf("1.5")
        )

    return mp.quad(tail_angle, [0, mp.pi / 2])


OFFICIAL_TRUE = [0.4839239, 0.1873623, 0.06993428, 0.02584440, 0.009521724]
OFFICIAL_PREDICTION = [0.4936814, 0.1811349, 0.06492889, 0.02339828, 0.008430809]
TRUE_TAIL = [float(harrison_true_tail(t)) for t in T_VALUES]
assert max(abs(a - b) for a, b in zip(TRUE_TAIL, OFFICIAL_TRUE)) < 6e-8


def mean_relative_error(prediction, truth):
    errors = [abs(prediction[i] - truth[i]) / truth[i] for i in range(len(truth))]
    return float(np.mean(errors)) if all(math.isfinite(value) for value in errors) else math.nan


def talbot_precision_audit(model):
    sample_node = mp.mpc("1.25", "0.75")
    node_tensor = torch.full((1, D), complex(sample_node), dtype=c_dtype, device=device)
    with torch.no_grad():
        log_phi = model(node_tensor)[0, 0]
        phi = torch.exp(log_phi).detach().cpu().item()
    low_precision = (
        node_tensor.dtype == torch.complex64
        or log_phi.dtype == torch.complex64
        or any(parameter.dtype in {torch.float32, torch.complex64} for parameter in model.parameters())
    )
    return {
        "model_parameter_dtypes": sorted({str(parameter.dtype) for parameter in model.parameters()}),
        "talbot_node_tensor_dtype": str(node_tensor.dtype),
        "model_log_output_dtype": str(log_phi.dtype),
        "python_output_component_types": [type(phi.real).__name__, type(phi.imag).__name__],
        "mpmath_version": mp.__version__,
        "mpmath_working_precision_dps": int(mp.mp.dps),
        "float32_or_complex64_cast_detected": bool(low_precision),
    }


def resolve_evaluation_dir():
    if RUN_MODE == "train":
        return FORMAL_RUN_DIR
    if RUN_MODE == "evaluate":
        if EVALUATE_RUN_DIR is None:
            raise ValueError("Set EVALUATE_RUN_DIR explicitly before using RUN_MODE='evaluate'.")
        return Path(EVALUATE_RUN_DIR).expanduser().resolve()
    return None


def evaluate_final_endpoint(run_dir):
    run_dir = Path(run_dir)
    checkpoint_path = run_dir / "checkpoint_step15000.pt"
    hash_before = sha256_file(checkpoint_path)
    payload = safe_torch_load(checkpoint_path)
    assert payload["experiment"] == EXPERIMENT
    assert payload["seed"] == SEED and payload["step"] == TRAIN_STEPS
    assert payload["model_config"] == MODEL_CONFIG
    assert payload["train_config"] == TRAIN_CONFIG

    model = build_model(PARAMS, SEED, payload["model_config"])
    model.load_state_dict(payload["model_state_dict"], strict=True)
    model.eval()
    assert parameter_counts(model) == PARAMETER_COUNTS

    precision = talbot_precision_audit(model)
    assert not precision["float32_or_complex64_cast_detected"]
    read_only_interaction = interaction_residual_audit(model, "read_only_evaluation_step_15000")
    assert read_only_interaction["interaction_residual_passed"]

    learned_zero_limit = model_tail_zero_limit(model)
    rows = []
    predictions = []
    evaluation_started = time.perf_counter()
    for t, truth in zip(T_VALUES, TRUE_TAIL):
        error_message = ""
        try:
            raw, imaginary, invalid = parse_raw_prediction(
                invert_tail(
                    lambda s: model_sum_laplace(model, s), t, learned_zero_limit
                )
            )
        except Exception as exc:
            raw, imaginary, invalid = math.nan, math.nan, True
            error_message = f"{type(exc).__name__}: {exc}"
        predictions.append(raw)
        rows.append({
            "t": t,
            "true_value": truth,
            "raw_prediction": raw,
            "raw_prediction_imaginary": imaginary,
            "relative_error": abs(raw - truth) / truth,
            "invalid_flag": invalid,
            "error_message": error_message,
        })

    summary = {
        "experiment": EXPERIMENT,
        "seed": SEED,
        "fixed_endpoint": TRAIN_STEPS,
        "talbot_degree": PRIMARY_TALBOT_DEGREE,
        "mre_1_5": mean_relative_error(predictions, TRUE_TAIL),
        "invalid_count": sum(row["invalid_flag"] for row in rows),
        "official_nn4mgf_mre_1_5": mean_relative_error(OFFICIAL_PREDICTION, TRUE_TAIL),
        "exact_transform_talbot_floor_status": "not_available",
        "exact_transform_talbot_floor_reason": (
            "The audited Harrison sources provide a density/tail integral but no reliable "
            "closed-form exact joint transform; no inversion floor is fabricated."
        ),
        "learned_tail_transform_zero_limit": float(learned_zero_limit),
        "nonseparability_interaction_residual_abs": read_only_interaction["maximum_interaction_residual_abs"],
        "nonseparability_audit_passed": read_only_interaction["interaction_residual_passed"],
        "raw_predictions": True,
        "prediction_clipping": False,
        "evaluation_wall_clock_seconds": time.perf_counter() - evaluation_started,
        "checkpoint_sha256": hash_before,
    }

    hash_after = sha256_file(checkpoint_path)
    assert hash_after == hash_before
    summary["checkpoint_hash_unchanged_after_evaluation"] = True
    write_csv(run_dir / "pointwise_degree5.csv", rows, list(rows[0]))
    baseline_rows = [
        {
            "t": t,
            "true_value": truth,
            "published_prediction": prediction,
            "relative_error": abs(prediction - truth) / truth,
        }
        for t, truth, prediction in zip(T_VALUES, TRUE_TAIL, OFFICIAL_PREDICTION)
    ]
    write_csv(run_dir / "published_nn4mgf_degree5.csv", baseline_rows, list(baseline_rows[0]))
    write_json(run_dir / "summary_degree5.json", summary)
    write_json(run_dir / "official_baseline_provenance.json", OFFICIAL_PROVENANCE)
    write_json(run_dir / "talbot_precision_audit.json", precision)

    nonseparability_path = run_dir / "nonseparability_audit.json"
    nonseparability = json.loads(nonseparability_path.read_text(encoding="utf-8"))
    nonseparability["read_only_evaluation"] = read_only_interaction
    write_json(nonseparability_path, nonseparability)

    fig, axis = plt.subplots(figsize=(7.2, 4.6))
    axis.semilogy(T_VALUES, TRUE_TAIL, "k-o", label="exact Harrison tail")
    axis.semilogy(T_VALUES, predictions, "C0-s", label="HCN, step 15k")
    axis.semilogy(T_VALUES, OFFICIAL_PREDICTION, "C1-^", label="published NN4MGF")
    axis.set(xlabel="t", ylabel="tail probability", title="2D Harrison non-product benchmark")
    axis.grid(True, alpha=0.3)
    axis.legend()
    fig.tight_layout()
    fig.savefig(run_dir / "tail_probability_degree5.png", dpi=180)
    plt.close(fig)

    print(json.dumps(summary, indent=2))
    return summary


EVALUATION_DIR = resolve_evaluation_dir()
FINAL_SUMMARY = None
if EVALUATION_DIR is not None:
    FINAL_SUMMARY = evaluate_final_endpoint(EVALUATION_DIR)
else:
    print("Smoke test complete. Set RUN_MODE='train' and Run All for the paper experiment.")
